# Phase 5 — Transfer Learning: Feature Extraction

Goal: an ImageNet-pretrained ResNet50 with the entire backbone frozen, training only
a fresh linear classifier head on top -- the first of three transfer-learning
variants (feature extraction, partial fine-tuning, full fine-tuning) compared
against Phase 4's from-scratch CNN in Phase 8. Uses Phase 2's fixed split and the
exact Phase 3 harness (`src/training/`), unmodified.

Preprocessing needs no changes here: Phase 2's transforms (224x224, ImageNet
mean/std normalization) were deliberately chosen to already match what a
torchvision ImageNet-pretrained backbone expects (`src/data/transforms.py`).

**Status:** authored and locally smoke-tested (imports, shapes, a couple of training
steps on a tiny subset, CPU-only here) -- the real 30-epoch run happens on Colab GPU,
same as Phase 4. Takeaways at the bottom get filled in with real numbers after that
run.

In [ ]:
# Colab only: clone (or pull) the GitHub repo and cd into it. Safe to run locally
# too -- the import fails there and this cell is skipped. Colab is pull-only (no
# push credentials here) -- see README's "Version control" section. Uses a plain
# git clone rather than Drive mount so this always picks up whatever was last
# pushed, instead of a manually-maintained Drive copy going stale.
try:
    import google.colab
    import os

    REPO_URL = "https://github.com/diljithG3/plant-disease-classification.git"
    REPO_DIR = "/content/plant-disease-classification"

    if os.path.isdir(REPO_DIR):
        !git -C {REPO_DIR} pull
    else:
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
except ImportError:
    pass

In [ ]:
import sys, os

# Walk up from cwd to the project root (marked by configs/config.yaml) and put it on
# sys.path. Plain os.getcwd() is not reliable here -- it depends on how the kernel was
# launched (VS Code's Jupyter extension defaults to the notebook's own folder, not the
# project root).
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "configs", "config.yaml")):
    _parent = os.path.dirname(_root)
    if _parent == _root:
        raise RuntimeError("Could not find project root (configs/config.yaml) above " + os.getcwd())
    _root = _parent
sys.path.append(_root)

from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.utils.device import get_device
from src.data import splits, transforms as T, dataset as D
from src.data.fetch_data import fetch_dataset
from src.models.transfer import build_resnet50_feature_extractor
from src.training.trainer import train_model
from src.training.checkpoint import load_checkpoint
from src.training import engine
from src.training.metrics import plot_confusion_matrix

cfg = load_config()
seed_everything(cfg["seed"])
device = get_device(cfg["training"]["device"])
print("Environment:", cfg["env"], "| Device:", device)

fetch_dataset(cfg)  # no-op if already downloaded

## 1. Load Phase 2's fixed split

Reuses the split and `class_to_idx` cached by `02_data_pipeline.ipynb` unchanged --
same requirement as Phase 4: recomputing a split here would break comparability with
every other phase (`CLAUDE.md`).

In [ ]:
DATA_PATH = cfg["data"]["path"]
ARTIFACTS_DIR = cfg["data"]["artifacts_dir"]

splits_path = Path(ARTIFACTS_DIR) / "splits.csv"
class_to_idx_path = Path(ARTIFACTS_DIR) / "class_to_idx.json"

if not splits_path.exists() or not class_to_idx_path.exists():
    raise FileNotFoundError(
        f"{splits_path} or {class_to_idx_path} missing -- run notebooks/02_data_pipeline.ipynb first."
    )

splits_df = pd.read_csv(splits_path)
class_to_idx = splits.build_class_to_idx(DATA_PATH, cache_path=str(class_to_idx_path))
idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)

print(f"Loaded split: {len(splits_df)} images, {num_classes} classes")
splits_df["split"].value_counts()

## 2. Dataloaders -- full dataset

Same fixed split, same transforms as every other phase. Meant to run on Colab GPU --
local is CPU-only, fine for the cells above but not for a real training run.

In [ ]:
train_transform = T.get_train_transforms(cfg)
eval_transform = T.get_eval_transforms(cfg)

loaders = D.make_dataloaders(
    splits_df,
    class_to_idx,
    train_transform,
    eval_transform,
    batch_size=cfg["dataloader"]["batch_size"],
    num_workers=cfg["dataloader"]["num_workers"],
    data_path=DATA_PATH,
)

for split, loader in loaders.items():
    print(f"{split}: {len(loader.dataset)} images, {len(loader)} batches")

## 3. Model, loss, optimizer

`build_resnet50_feature_extractor` (`src/models/transfer.py`): torchvision's
ImageNet-pretrained ResNet50 with every backbone parameter frozen
(`requires_grad=False`), and its final `fc` layer replaced with a fresh linear head
for this dataset's 38 classes -- only the head is trained.

Loss is unweighted `CrossEntropyLoss`, matching Phase 4's baseline choice, so any
accuracy/macro-F1 difference from Phase 4 reflects the backbone change, not a
different imbalance-handling strategy. The optimizer is built from
`filter(requires_grad)` over the model's parameters -- i.e. just `fc` -- since
backpropagating through the frozen backbone would be wasted compute.

In [ ]:
model = build_resnet50_feature_extractor(num_classes).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} total (backbone frozen, only `fc` trains)")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

## 4. Train

`max_epochs=30` and `early_stopping_patience: 7` on `macro_f1`
(`configs/config.yaml`) -- identical settings to every Phase 4 run, so only the
model differs.

In [ ]:
checkpoints_dir = Path(cfg["logging"]["checkpoints_dir"]) / "phase5_resnet50_feature_extraction"
resume_path = checkpoints_dir / "last.pt"
if resume_path.exists():
    print(f"Found checkpoint from an earlier attempt, resuming from {resume_path}")

history = train_model(
    model, loaders, criterion, optimizer, device,
    run_name="phase5_resnet50_feature_extraction", cfg=cfg, class_to_idx=class_to_idx, max_epochs=30,
    resume_from=resume_path if resume_path.exists() else None,
)
pd.DataFrame(history)

## 5. Verify checkpoint round-trip, then evaluate on the held-out test set

Same round-trip check as every previous phase: confirm `best.pt` reloads into a
**fresh** model instance and reproduces the metric it was saved with -- then use
that reloaded model for the real held-out test-set evaluation (not the val set used
during training/early stopping, which already influenced checkpoint selection).

In [ ]:
checkpoints_dir = Path(cfg["logging"]["checkpoints_dir"]) / "phase5_resnet50_feature_extraction"
best_path = checkpoints_dir / "best.pt"
assert best_path.exists()

fresh_model = build_resnet50_feature_extractor(num_classes).to(device)
payload = load_checkpoint(best_path, fresh_model, optimizer=None, map_location=device)
assert payload["class_to_idx"] == class_to_idx, "class_to_idx did not round-trip!"
print(f"Reloaded checkpoint from epoch {payload['epoch']}, best val {cfg['training']['checkpoint_metric']}={payload['best_metric']:.4f}")

test_metrics = engine.evaluate(fresh_model, loaders["test"], criterion, device, num_classes)
print(f"Test accuracy: {test_metrics['accuracy']:.4f} | Test macro_f1: {test_metrics['macro_f1']:.4f}")

## 6. Confusion matrix (test set)

In [ ]:
class_names = [idx_to_class[i] for i in range(num_classes)]
plot_confusion_matrix(test_metrics["confusion_matrix"], class_names)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

## 7. Per-class recall vs. train-set frequency

Same check Phase 4 ran on its baseline: does the 36x class imbalance show up in
which classes this model struggles with, or does frozen ImageNet feature reuse
sidestep the pattern Phase 4 found (same-species disease confusion, not raw class
frequency)?

In [ ]:
recall_series = pd.Series(
    test_metrics["per_class_recall"], index=[idx_to_class[i] for i in range(num_classes)]
).sort_values()

train_counts = splits_df[splits_df["split"] == "train"]["class"].value_counts()
comparison = pd.DataFrame({"train_count": train_counts, "test_recall": recall_series}).sort_values("train_count")

print("Lowest test recall (worst-performing classes):")
print(recall_series.head(10))
comparison

## Summary & implications for Phase 6

In [ ]:
print(f"Feature-extraction test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Feature-extraction test macro_f1: {test_metrics['macro_f1']:.4f}")
print(f"Worst-recall class: {recall_series.index[0]} (recall={recall_series.iloc[0]:.3f}, train_count={int(train_counts.get(recall_series.index[0], 0))})")
print(f"Best-recall class: {recall_series.index[-1]} (recall={recall_series.iloc[-1]:.3f})")